# Model Comparison & Conclusion

This notebook loads all 9 trained model checkpoints, evaluates them on the test set, and produces a final comparison table with conclusions.

## 1. Setup

In [ ]:
# Standard library
import os
import random
from pathlib import Path

# Third-party
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix

# TensorFlow / Keras
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.regularizers import L2
import keras_tuner as kt

print(f"TensorFlow version: {tf.__version__}")
print(f"GPU available: {tf.config.list_physical_devices('GPU')}")

In [ ]:
# ============================================================
# GLOBAL CONFIGURATION
# ============================================================

DATASET_PATH = Path(r"C:\Users\Owent\Desktop\ODL_assg\oral")

SEED = 42
IMAGE_SIZE = (224, 224)
BATCH_SIZE = 32
NUM_CLASSES = 6
EPOCHS_BASELINE = 100      # Custom CNN improved configs
EPOCHS_PRETRAINED = 50     # Transfer learning improved configs
EPOCHS_TUNER = 30          # Keras Tuner trials
EPOCHS_REPLICA = 20        # Replica configs (matching reference)

# Set random seeds for reproducibility
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

# Create output directories
os.makedirs("checkpoints", exist_ok=True)
os.makedirs("logs", exist_ok=True)

print("Configuration set.")

## 2. Data Pipeline

In [ ]:
def load_dataset(dataset_path: Path) -> pd.DataFrame:
    """Walk dataset directory, collect image paths and labels. Skips yolo folders."""
    if not dataset_path.exists():
        raise FileNotFoundError(f"Dataset path does not exist: {dataset_path}")
    valid_extensions = {'.jpg', '.jpeg', '.png', '.bmp'}
    records = []
    class_folders = [f for f in dataset_path.iterdir() if f.is_dir()]
    for class_folder in sorted(class_folders):
        class_name = class_folder.name
        if 'yolo' in class_name.lower():
            print(f"  [SKIPPED] {class_name}")
            continue
        class_count = 0
        for root, dirs, files in os.walk(class_folder):
            dirs[:] = [d for d in dirs if 'yolo' not in d.lower()]
            for fname in files:
                ext = os.path.splitext(fname)[1].lower()
                if ext in valid_extensions:
                    filepath = os.path.join(root, fname)
                    records.append({'filepath': filepath, 'label': class_name})
                    class_count += 1
        if class_count == 0:
            print(f"  [WARNING] No images in: {class_name}")
        else:
            print(f"  [OK] {class_name}: {class_count} images")
    if len(records) == 0:
        raise ValueError("No valid images found.")
    return pd.DataFrame(records)

# Load dataset
df = load_dataset(DATASET_PATH)
CLASS_NAMES = sorted(df['label'].unique().tolist())
LABEL_TO_INDEX = {name: idx for idx, name in enumerate(CLASS_NAMES)}
print(f"\nTotal: {len(df)} images, Classes: {CLASS_NAMES}")

# Stratified 70/15/15 split
train_df, temp_df = train_test_split(df, test_size=0.30, random_state=47, stratify=df['label'])
val_df, test_df = train_test_split(temp_df, test_size=0.50, random_state=47, stratify=temp_df['label'])
train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)
print(f"Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}")

In [ ]:
AUTOTUNE = tf.data.AUTOTUNE

def process_path(filepath, label):
    """Read, decode, resize, normalize image; one-hot encode label."""
    img = tf.io.read_file(filepath)
    img = tf.io.decode_jpeg(img, channels=3)
    img = tf.image.resize(img, IMAGE_SIZE)
    img = tf.cast(img, tf.float32) / 255.0
    label = tf.one_hot(label, depth=NUM_CLASSES)
    return img, label

def build_dataset(dataframe, is_training=False):
    """Build batched, prefetched tf.data.Dataset from DataFrame."""
    filepaths = dataframe['filepath'].values
    labels = dataframe['label'].map(LABEL_TO_INDEX).values.astype(np.int32)
    ds = tf.data.Dataset.from_tensor_slices((filepaths, labels))
    ds = ds.map(process_path, num_parallel_calls=AUTOTUNE)
    if is_training:
        ds = ds.shuffle(buffer_size=1000)
    ds = ds.batch(BATCH_SIZE)
    ds = ds.prefetch(AUTOTUNE)
    return ds

train_ds = build_dataset(train_df, is_training=True)
val_ds = build_dataset(val_df, is_training=False)
test_ds = build_dataset(test_df, is_training=False)
print("Datasets built successfully.")

## 3. Load Training Histories

Load CSV logs saved during training to reconstruct plots if needed.

In [ ]:
# Load all training history CSVs
import glob

history_files = sorted(glob.glob("logs/*_history.csv"))
print(f"Found {len(history_files)} history files:")
for f in history_files:
    print(f"  {f}")

## 4. Rebuild & Evaluate All Models

Rebuild each architecture, load best checkpoint weights, evaluate on test set.

In [ ]:
# --- Helper to evaluate a model from checkpoint ---
def quick_eval(model, checkpoint_path, model_name):
    """Load weights and evaluate."""
    model.load_weights(checkpoint_path)
    train_loss, train_acc = model.evaluate(train_ds, verbose=0)
    val_loss, val_acc = model.evaluate(val_ds, verbose=0)
    test_loss, test_acc = model.evaluate(test_ds, verbose=0)
    return {
        'Model': model_name.split(' ')[0],
        'Config': model_name.split(' ')[-1],
        'Params (M)': round(model.count_params() / 1e6, 2),
        'Train Acc (%)': round(train_acc * 100, 2),
        'Val Acc (%)': round(val_acc * 100, 2),
        'Test Acc (%)': round(test_acc * 100, 2),
        'Train Loss': round(train_loss, 4),
        'Val Loss': round(val_loss, 4),
        'Test Loss': round(test_loss, 4),
    }

all_results = []

In [ ]:
# --- CNN C1 ---
cnn_c1 = keras.Sequential([
    keras.Input(shape=(224, 224, 3)),
    layers.Conv2D(32, (3, 3), activation="relu"),
    layers.MaxPooling2D(pool_size=(2, 2)),
    layers.Conv2D(64, (3, 3), activation="relu"),
    layers.MaxPooling2D(pool_size=(2, 2)),
    layers.Conv2D(128, (3, 3), activation="relu"),
    layers.MaxPooling2D(pool_size=(2, 2)),
    layers.GlobalAveragePooling2D(),
    layers.Dense(128, activation="relu"),
    layers.Dropout(0.5),
    layers.Dense(NUM_CLASSES, activation="softmax")
])
cnn_c1.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
all_results.append(quick_eval(cnn_c1, "checkpoints/cnn_c1_best.h5", "Custom_CNN C1"))
print("CNN C1 evaluated.")

In [ ]:
# --- CNN C2 (rebuild architecture) ---
def _build_cnn_c2():
    inputs = keras.Input(shape=(224, 224, 3))
    x = keras.Sequential([layers.RandomFlip("horizontal"), layers.RandomRotation(0.05)])(inputs)
    for filters, drop in [(64, 0.2), (128, 0.3), (256, 0.4), (256, 0.4)]:
        x = layers.Conv2D(filters, 3, activation='relu', padding='same', kernel_regularizer=L2(1e-4))(x)
        x = layers.Conv2D(filters, 3, activation='relu', padding='same', kernel_regularizer=L2(1e-4))(x)
        x = layers.BatchNormalization()(x)
        x = layers.MaxPooling2D(2)(x)
        x = layers.Dropout(drop)(x)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(1024, activation='relu', kernel_regularizer=L2(1e-4))(x)
    x = layers.Dropout(0.5)(x)
    x = layers.Dense(512, activation='relu', kernel_regularizer=L2(1e-4))(x)
    x = layers.Dropout(0.3)(x)
    outputs = layers.Dense(NUM_CLASSES, activation='softmax')(x)
    model = keras.Model(inputs, outputs)
    model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
    return model

cnn_c2 = _build_cnn_c2()
all_results.append(quick_eval(cnn_c2, "checkpoints/cnn_c2_best.h5", "Custom_CNN C2"))
print("CNN C2 evaluated.")

In [ ]:
# --- CNN C3 (load from tuner or checkpoint) ---
# Note: C3 architecture varies per tuning. If checkpoint exists, we reload.
# For comparison we just report from the saved history.
try:
    # Attempt to load — architecture must match what was saved
    cnn_c3_hist = pd.read_csv("logs/cnn_c3_history.csv")
    best_test = cnn_c3_hist['val_accuracy'].max()
    print(f"CNN C3 best val_accuracy from logs: {best_test:.4f}")
    print("  (Full eval requires rebuilding exact tuner architecture — see notebook 02)")
except FileNotFoundError:
    print("CNN C3 history not found. Run notebook 02 first.")

In [ ]:
# --- EfficientNetB0 C1 ---
base_eff = keras.applications.EfficientNetB0(weights="imagenet", include_top=False, input_shape=(224, 224, 3))
base_eff.trainable = False
eff_c1 = keras.Sequential([
    keras.Input(shape=(224, 224, 3)), base_eff,
    layers.GlobalAveragePooling2D(), layers.Dense(128, activation="relu"),
    layers.Dropout(0.5), layers.Dense(NUM_CLASSES, activation="softmax")
])
eff_c1.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
all_results.append(quick_eval(eff_c1, "checkpoints/eff_c1_best.h5", "EfficientNetB0 C1"))
print("EfficientNetB0 C1 evaluated.")

In [ ]:
# --- ResNet50 C1 ---
base_res = keras.applications.ResNet50(weights="imagenet", include_top=False, input_shape=(224, 224, 3))
base_res.trainable = False
res_c1 = keras.Sequential([
    keras.Input(shape=(224, 224, 3)), base_res,
    layers.GlobalAveragePooling2D(), layers.Dense(128, activation="relu"),
    layers.Dropout(0.5), layers.Dense(NUM_CLASSES, activation="softmax")
])
res_c1.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
all_results.append(quick_eval(res_c1, "checkpoints/res_c1_best.h5", "ResNet50 C1"))
print("ResNet50 C1 evaluated.")

## 5. Comparison Table

In [ ]:
# Build comparison table
comp_df = pd.DataFrame(all_results)

# Highlight best test accuracy
max_test = comp_df['Test Acc (%)'].max()

def highlight_best(row):
    if row['Test Acc (%)'] == max_test:
        return ['background-color: yellow; font-weight: bold'] * len(row)
    return [''] * len(row)

styled = comp_df.style.apply(highlight_best, axis=1)
print("\n" + "="*60)
print("          FINAL MODEL COMPARISON (9 configurations)")
print("="*60)
display(styled)

# Also print as plain text
print(comp_df.to_string(index=False))

## 6. Conclusion

### Best Configuration Per Model Family

| Model Family | Best Config | Test Accuracy |
|---|---|---|
| Custom CNN | *(fill after training)* | *(fill)* |
| EfficientNetB0 | *(fill after training)* | *(fill)* |
| ResNet50 | *(fill after training)* | *(fill)* |

### Recommended Model for Deployment

*(Update after all experiments complete. Justify using test accuracy + parameter count or validation loss.)*

### Future Improvements

1. **More aggressive augmentation** — CutMix, MixUp, or elastic deformations for better generalization.
2. **Additional architectures** — DenseNet121, ConvNeXt-Tiny, or Vision Transformers (ViT-Small).
3. **Class balancing** — Weighted loss or oversampling for underrepresented classes.
4. **Ensemble methods** — Combine top models via soft voting.
5. **External data** — Collect more images for minority classes.